## 1. Import các thư viện và module cần thiết


In [1]:
import torch
import sentencepiece as spm
import pandas as pd
from tqdm.notebook import tqdm
import evaluate
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_colwidth', None)

from inference import beam_search_decode, summarize_with_beam_search

from utils.config import get_config
from utils.data_loader import load_data_from_csv, collate_fn
from utils.dataset import VietNewSumDataset
from torch.utils.data import DataLoader
from model.transformer import build_transformer

d:\viet_summarizer\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Cấu hình các đường dẫn


In [2]:
TEST_CSV_PATH = "data/processed/small_test.csv"
TRAIN_CSV_PATH = "data/processed/train.csv"
TOKENIZER_PATH = "tokenizers/spm_vietnamese_model.model"
CHECKPOINT_DIR = "checkpoints"

config = get_config()
config['test_csv'] = TEST_CSV_PATH
config['train_csv'] = TRAIN_CSV_PATH
config['tokenizer_file'] = TOKENIZER_PATH
config['save_dir'] = CHECKPOINT_DIR
config['batch_size'] = 1


## 3. Tải Model, Tokenizer và Dữ liệu


In [3]:
def get_model_and_tokenizer(config):
    """Tải model và tokenizer từ checkpoint."""
    device = config['device']
    
    tokenizer = spm.SentencePieceProcessor()
    tokenizer.load(config["tokenizer_file"])

    model = build_transformer(
        src_vocab_size=tokenizer.get_piece_size(),
        tgt_vocab_size=tokenizer.get_piece_size(),
        src_seq_len=config["max_len_text"],
        tgt_seq_len=config["max_len_summary"],
        d_model=config["d_model"],
        N=config["num_layers"],
        h=config["num_heads"],
        d_ff=config["d_ff"],
        dropout=config["dropout"],
    ).to(device)
    
    model_path = Path(config["save_dir"]) / "transformer_summarizer_best.pt"
    state = torch.load(model_path, map_location=device)
    
    state_dict = state['model_state_dict']
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('_orig_mod.'):
            new_state_dict[k[len('_orig_mod.'):]] = v
        else:
            new_state_dict[k] = v
    
    model.load_state_dict(new_state_dict)
    
    return model, tokenizer

In [4]:
try:
    print("Đang tải model và tokenizer...")
    model, tokenizer = get_model_and_tokenizer(config)
    device = config['device']
    model.eval()
    print("Tải model và tokenizer thành công!")

    _, _, test_df = load_data_from_csv(config['train_csv'], None, config['test_csv'])
    
    # Tạo DataLoader
    test_dataset = VietNewSumDataset(
        dataframe=test_df, tokenizer=tokenizer,
        max_len_text=config['max_len_text'], max_len_summary=config['max_len_summary'],
        truncate_strategy_text=config.get('truncate_strategy_text', 'head')
    )
    test_dataloader = DataLoader(
        dataset=test_dataset, batch_size=1, shuffle=False,
        collate_fn=collate_fn, num_workers=2
    )
    print(f"Đã tạo DataLoader cho {len(test_df)} mẫu test.")
    
except FileNotFoundError as e:
    print(f"Lỗi: {e}. Vui lòng kiểm tra lại các đường dẫn.")
    model = None
    test_dataloader = None


Đang tải model và tokenizer...
Tải model và tokenizer thành công!
Đã tạo DataLoader cho 20 mẫu test.


## 4. Chạy Đánh giá


In [5]:
if model and test_dataloader:
    source_texts = []
    predicted_summaries = []
    reference_summaries = []

    with torch.no_grad():
        print("Đang đánh giá trên tập test.............")
        for batch in test_dataloader:
            encoder_input = batch['encoder_input'].to(device)
            encoder_mask = batch['encoder_mask'].to(device)
            
            # Sử dụng hàm greedy_decode đã import từ inference.py
            model_out_ids = beam_search_decode(
                model,
                encoder_input,
                encoder_mask,
                tokenizer,
                config['max_len_summary'],
                device,
                beam_width=5,
                temperature=0.9
            )
            
            # Chuyển đổi ID sang văn bản
            predicted_text = tokenizer.decode(model_out_ids.cpu().numpy().tolist())
            
            # Lưu lại tất cả kết quả
            source_texts.append(batch['src_text'][0])
            predicted_summaries.append(predicted_text)
            reference_summaries.append(batch['tgt_text'][0])

    print("\\nĐánh giá hoàn tất!")
    
    results_df = pd.DataFrame({
        'Source Text': source_texts,
        'Reference Summary': reference_summaries,
        'Predicted Summary': predicted_summaries
    })

else:
    print("Bỏ qua đánh giá do model hoặc dataloader chưa được tải thành công.")

Đang đánh giá trên tập test.............
\nĐánh giá hoàn tất!


## 5. Tính toán ROUGE Score và Hiển thị Kết quả


In [ ]:
if 'results_df' in locals() and not results_df.empty:
    print("📊 Đang tính toán điểm ROUGE...")
    try:
        rouge_metric = evaluate.load('rouge')
        
        rouge_scores = rouge_metric.compute(
            predictions=results_df['Predicted Summary'].tolist(),
            references=results_df['Reference Summary'].tolist()
        )
        
        # In kết quả
        print("\n" + "="*50)
        print("   KẾT QUẢ ĐÁNH GIÁ ĐỊNH LƯỢNG (ROUGE SCORES)")
        print("="*50)
        print(f"  Tổng số mẫu: {len(results_df)}")
        print("-" * 50)
        print(f"  ROUGE-1 F1 : {rouge_scores.get('rouge1', 0) * 100:.2f}%")
        print(f"  ROUGE-2 F1 : {rouge_scores.get('rouge2', 0) * 100:.2f}%")
        print(f"  ROUGE-L F1 : {rouge_scores.get('rougeL', 0) * 100:.2f}%")
        print("="*50)

    except Exception as e:
        print(f"Lỗi khi tính điểm ROUGE: {e}")

    print("\n\n" + "="*50)
    print("      PHÂN TÍCH ĐỊNH TÍNH (VÍ DỤ NGẪU NHIÊN)")
    print("="*50)
    
    # Lấy 3 mẫu ngẫu nhiên để xem xét
    num_samples_to_show = min(3, len(results_df))
    sample_view = results_df.sample(n=num_samples_to_show, random_state=42)
    
    print(f"Hiển thị {num_samples_to_show} mẫu ngẫu nhiên:")
    display(sample_view)

    output_file = "evaluation_results.csv"
    try:
        results_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    except Exception as e:
        print(f"❌ Lỗi khi lưu file: {e}")

else:
    print("Không tìm thấy 'results_df'. Vui lòng chạy cell đánh giá để tạo kết quả trước.")


📊 Đang tính toán điểm ROUGE...

   KẾT QUẢ ĐÁNH GIÁ ĐỊNH LƯỢNG (ROUGE SCORES)
  Tổng số mẫu: 20
--------------------------------------------------
  ROUGE-1 F1 : 43.17%
  ROUGE-2 F1 : 14.36%
  ROUGE-L F1 : 28.86%


      PHÂN TÍCH ĐỊNH TÍNH (VÍ DỤ NGẪU NHIÊN)
Hiển thị 3 mẫu ngẫu nhiên:


Source Text  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 tiền phong đưa tin , vụ cháy xảy ra vào tối 10/10 . người dân phát hiện lửa , khói bốc lên tại khu vực nhà xưởng sản xuất đồ chơi trẻ em thuộc địa phận phường việt hưng , quận long biên ( hà nội ) đã nhanh chóng báo lên cơ quan chức năng . lực lượng cảnh sát pccc cnch quận long biên đã huy động 2 xe cứu hoả chuyên dụng cùng nhiều cán bộ chiến sĩ đến hiện trường dập lửa . thông tin trên báo kiến thức , do lửa cháy lớn nên công tác cứu hoả ban đầu gặp rất nhiều



      LƯU KẾT QUẢ
